In [ ]:
# import requests
# import re
# from bs4 import BeautifulSoup
# import time

# print("=== Zone-H Grabber (Refactor) ===")

# # nhập cookie
# zhe = input("Nhập cookie ZHE: ")
# phpsessid = input("Nhập cookie PHPSESSID: ")

# cookie = {
#     "ZHE": zhe,
#     "PHPSESSID": phpsessid
# }

# # nhập số trang
# num_pages = int(input("Nhập số trang notifier cần lấy: "))
# num_pages_per_notifier = int(input("Nhập số trang mỗi notifier: "))

# notiferz = []

# print("Đang lấy danh sách notifier...")

# for n in range(num_pages):
#     url = f'https://zone-h.org/archive/published=0/page={n+1}'
#     usr = requests.get(url, cookies=cookie).content

#     # check captcha
#     if 'captcha' in usr.decode('utf-8').lower():
#         input("Bị captcha → mở web verify rồi nhấn Enter...")
#         usr = requests.get(url, cookies=cookie).content

#     soup = BeautifulSoup(usr, 'html.parser')

#     # lấy notifier đúng cách (không parse string bẩn)
#     for a in soup.find_all('a', href=True):
#         if '/archive/notifier=' in a['href']:
#             notif = a['href'].split('=')[1]
#             if notif not in notiferz:
#                 notiferz.append(notif)
#                 with open('notiferz.txt', 'a+') as f:
#                     f.write(notif + '\n')

#     time.sleep(1)

# print(f"Tổng notifier: {len(notiferz)}")

# sitez = []

# print("Đang lấy danh sách site...")

# for notifier in notiferz:
#     print(f"Đang xử lý: {notifier}")

#     for j in range(num_pages_per_notifier):
#         url = f'http://www.zone-h.org/archive/notifier={notifier}/page={j+1}'
#         verif = requests.get(url, cookies=cookie).content

#         if 'captcha' in verif.decode('utf-8').lower():
#             input("Bị captcha → verify rồi nhấn Enter...")
#             verif = requests.get(url, cookies=cookie).content

#         soup = BeautifulSoup(verif, 'html.parser')

#         # check hết dữ liệu
#         check = soup.find("td", {"class": "defacepages"})
#         if check and "<strong>0</strong>" in str(check):
#             break

#         html = verif.decode('utf-8')

#         # vẫn dùng regex nhưng đã kiểm soát tốt hơn
#         king = re.findall(r'<td>(.*?)\n', html)

#         for oo in king:
#             domain = oo.split('/')[0]
#             newurl = 'http://' + domain

#             if newurl not in sitez:
#                 sitez.append(newurl)

#                 with open('allsites.txt', 'a+') as f:
#                     f.write(newurl + '\n')

#                 print(newurl)

#         time.sleep(1)

# print("Hoàn thành!")

In [ ]:
# import requests
# import re
# import sys
# import os
# import platform

# # Nhận cookies
# zhe_cookie = input("\nEnter ZHE cookie: ").strip()
# phpsessid_cookie = input("Enter PHPSESSID cookie: ").strip()

# cookie = {
#     "ZHE": zhe_cookie,
#     "PHPSESSID": phpsessid_cookie
# }

# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 6.1; rv:57.0) Gecko/20100101 Firefox/57.0"
# }

# def grab_archive_sites():
#     for i in range(1, 51):
#         url = f"http://www.zone-h.org/archive/special={i}"
#         res = requests.get(url, cookies=cookie, headers=headers)
#         html = res.content

#         print(f"\n Đang lấy từ: {url}")

#         if b'captcha' in html:
#             print("!!! CAPTCHA detected. Hãy xác minh thủ công trên trình duyệt!")
#             sys.exit()

#         matches = re.findall(b'<td>([^<\n]+)\n\s*</td>', html)

#         with open("onhold_zone.txt", "a") as f:
#             for m in matches:
#                 domain = m.split(b'/')[0].decode().strip()
#                 print(f"[+] {domain}")
#                 f.write(f"http://{domain}\n")

# def main():
#     grab_archive_sites()

# if __name__ == "__main__":
#     main()

In [ ]:
import requests
import re
import sys
import time

# Nhập cookies
zhe_cookie = input("\nEnter ZHE cookie: ").strip()
phpsessid_cookie = input("Enter PHPSESSID cookie: ").strip()

cookie = {
    "ZHE": zhe_cookie,
    "PHPSESSID": phpsessid_cookie
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

def grab_archive_sites():
    for i in range(1, 100): # chỉnh số lượng lấy ở đây để lấy nhiều urls hơn
        url = f"https://www.zone-h.org/archive/special={i}"

        print(f"\nĐang lấy: {url}")

        try:
            res = requests.get(
                url,
                cookies=cookie,
                headers=headers,
                timeout=10
            )

        except requests.exceptions.ConnectTimeout:
            print("Timeout khi kết nối. Thử lại sau 5s...")
            time.sleep(5)
            continue

        except requests.exceptions.RequestException as e:
            print(f"Lỗi request: {e}")
            continue

        if res.status_code != 200:
            print(f"Status code lỗi: {res.status_code}")
            continue

        html = res.content

        # Detect CAPTCHA
        if b'captcha' in html.lower():
            print("\n!!! CAPTCHA detected !!!")
            print("1. Mở link này trên trình duyệt:")
            print(url)
            print("2. Giải CAPTCHA")
            print("3. Copy lại cookie mới (ZHE + PHPSESSID)")
            input("\nSau khi xong, nhấn Enter để tiếp tục...")

            # cập nhật cookie mới
            cookie["ZHE"] = input("Nhập lại ZHE cookie: ").strip()
            cookie["PHPSESSID"] = input("Nhập lại PHPSESSID cookie: ").strip()

            continue

        matches = re.findall(b'<td>([^<\n]+)\n\s*</td>', html)

        if not matches:
            print("Không tìm thấy dữ liệu")
            continue

        with open("urls.txt", "a", encoding="utf-8") as f:
            for m in matches:
                try:
                    domain = m.split(b'/')[0].decode(errors="ignore").strip()
                    print(f"[+] {domain}")
                    f.write(f"http://{domain}\n")
                except:
                    continue

        time.sleep(2)

def main():
    grab_archive_sites()

if __name__ == "__main__":
    main()


Đang lấy: https://www.zone-h.org/archive/special=1
[+] apmt.munisanmiguelpetapa.gob.g...
[+] siggec.gov.ao
[+] revistas.ufrj.br
[+] periodicos.ufal.br
[+] revistas.face.ufmg.br
[+] rbpg.capes.gov.br
[+] mcstaging.store.lenovo.com
[+] bimtek2.kpud-hulusungaitengahk...
[+] home.blog.kpud-hulusungaitenga...
[+] kab-pacitan.kpu.go.id.kpud-pac...
[+] apps.telessaude.hc.ufmg.br
[+] staging.transparencia.gob.pe
[+] plataformamincu.cultura.gob.pe...
[+] mpr.kpu-banglikab.go.id
[+] dprd.kpu-banglikab.go.id
[+] ppid.bukittinggi.bawaslu.go.id...
[+] bukittinggi.bawaslu.go.id
[+] kpu.takalarkab.go.id
[+] kesbangpol.takalarkab.go.id
[+] dprd.takalarkab.go.id
[+] commandcenter.bimakota.go.id
[+] genform.navy.mil.bd
[+] dinsos.kedirikab.go.id
[+] campoformoso.ba.gov.br
[+] www.kejari-sleman.go.id

Đang lấy: https://www.zone-h.org/archive/special=2
[+] thebloomingstoryindia.com
[+] rompetesgrow.com.ar
[+] behalinternational.com
[+] faminvestment.ae
[+] fastbooks.info
[+] espb.ao
[+] lamphandinh.com
[